# ⚡ Treinamento de Modelos - XGBoost

## Objetivo
Este notebook foca no treinamento e otimização do modelo XGBoost para previsão de casos de dengue.

## Estratégia
- Utilizar os mesmos dados preparados
- Otimização específica para XGBoost com Bayesian Optimization
- Comparação com Random Forest
- Análise de importância das features
- Early stopping para evitar overfitting

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

import xgboost as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import uniform, randint
import joblib
import warnings

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("✅ Bibliotecas carregadas com sucesso!")

In [ ]:
# Carregamento dos dados processados
print("📂 Carregando dados processados...")

df = pd.read_csv('dados_com_features.csv')
X = pd.read_csv('X_features.csv')
y = pd.read_csv('y_target.csv')['Quantidade_Casos']

# Recriar coluna de data para divisão temporal
df['Data'] = pd.to_datetime(df[['Ano', 'Mês']].assign(day=1))

print(f"📊 Dataset: {X.shape[0]:,} registros, {X.shape[1]} features")
print(f"🎯 Target: {y.shape[0]:,} valores")

In [ ]:
# Função de divisão temporal (mesma do notebook anterior)
def split_time_series_data(df, X, y, train_end='2021-12', val_end='2023-12'):
    """
    Divide os dados de forma temporal para evitar data leakage
    """
    train_mask = df['Data'] <= pd.to_datetime(train_end)
    val_mask = (df['Data'] > pd.to_datetime(train_end)) & (df['Data'] <= pd.to_datetime(val_end))
    test_mask = df['Data'] > pd.to_datetime(val_end)

    X_train = X[train_mask]
    y_train = y[train_mask]
    X_val = X[val_mask]
    y_val = y[val_mask]
    X_test = X[test_mask]
    y_test = y[test_mask]

    return X_train, X_val, X_test, y_train, y_val, y_test, train_mask, val_mask, test_mask

# Função para calcular métricas
def calculate_metrics(y_true, y_pred, model_name="Modelo"):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100

    return {
        'Modelo': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'MAPE': mape
    }

# Dividir dados
X_train, X_val, X_test, y_train, y_val, y_test, train_mask, val_mask, test_mask = split_time_series_data(df, X, y)

print(f"📊 Divisão dos dados:")
print(f"   🏋️ Treino: {X_train.shape[0]:,} registros")
print(f"   🎯 Validação: {X_val.shape[0]:,} registros")
print(f"   🧪 Teste: {X_test.shape[0]:,} registros")

In [ ]:
# Modelo baseline XGBoost
print("⚡ Treinando modelo baseline XGBoost...")

# Modelo baseline com parâmetros padrão
xgb_baseline = xgb.XGBRegressor(
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# Treinamento com early stopping
xgb_baseline.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=50,
    verbose=False
)

# Predições
y_pred_train_baseline = xgb_baseline.predict(X_train)
y_pred_val_baseline = xgb_baseline.predict(X_val)

# Métricas
metrics_train_baseline = calculate_metrics(y_train, y_pred_train_baseline, "XGB Baseline - Treino")
metrics_val_baseline = calculate_metrics(y_val, y_pred_val_baseline, "XGB Baseline - Validação")

print("📊 Métricas do modelo baseline:")
print("\n🏋️ Treino:")
for key, value in metrics_train_baseline.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🎯 Validação:")
for key, value in metrics_val_baseline.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print(f"\n🛑 Early stopping em: {xgb_baseline.best_iteration} iterações")

In [ ]:
# Otimização de hiperparâmetros com RandomizedSearch (mais eficiente que GridSearch)
print("🔧 Iniciando otimização de hiperparâmetros...")

# Parâmetros para otimização
param_distributions = {
    'n_estimators': randint(100, 1000),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 2)
}

print(f"🔍 Testando otimização com RandomizedSearch...")

# Cross-validation temporal
tscv = TimeSeriesSplit(n_splits=3)

# RandomizedSearch
xgb_model = xgb.XGBRegressor(
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions,
    n_iter=50,  # 50 iterações para balancear tempo e performance
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit na combinação treino + validação para otimização
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

random_search.fit(X_train_val, y_train_val)

print("✅ Otimização concluída!")
print(f"\n🏆 Melhores hiperparâmetros:")
for param, value in random_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Melhor score (neg_MSE): {random_search.best_score_:.4f}")

In [ ]:
# Modelo otimizado com early stopping
print("🚀 Treinando modelo otimizado...")

# Melhor modelo com parâmetros otimizados
best_params = random_search.best_params_.copy()
xgb_optimized = xgb.XGBRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# Treinamento com early stopping
xgb_optimized.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=['train', 'val'],
    early_stopping_rounds=100,
    verbose=False
)

# Predições
y_pred_train_opt = xgb_optimized.predict(X_train)
y_pred_val_opt = xgb_optimized.predict(X_val)
y_pred_test_opt = xgb_optimized.predict(X_test)

# Métricas
metrics_train_opt = calculate_metrics(y_train, y_pred_train_opt, "XGB Otimizado - Treino")
metrics_val_opt = calculate_metrics(y_val, y_pred_val_opt, "XGB Otimizado - Validação")
metrics_test_opt = calculate_metrics(y_test, y_pred_test_opt, "XGB Otimizado - Teste")

print("📊 Métricas do modelo otimizado:")
print("\n🏋️ Treino:")
for key, value in metrics_train_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🎯 Validação:")
for key, value in metrics_val_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

print("\n🧪 Teste:")
for key, value in metrics_test_opt.items():
    if key != 'Modelo':
        print(f"   {key}: {value:.4f}")

# Comparação com baseline
print(f"\n📈 Melhoria na validação:")
print(f"   RMSE: {metrics_val_baseline['RMSE']:.4f} → {metrics_val_opt['RMSE']:.4f} ({((metrics_val_baseline['RMSE'] - metrics_val_opt['RMSE'])/metrics_val_baseline['RMSE']*100):+.2f}%)")
print(f"   R²: {metrics_val_baseline['R²']:.4f} → {metrics_val_opt['R²']:.4f} ({((metrics_val_opt['R²'] - metrics_val_baseline['R²'])/metrics_val_baseline['R²']*100):+.2f}%)")

print(f"\n🛑 Early stopping em: {xgb_optimized.best_iteration} iterações")

In [ ]:
# Curva de aprendizado
print("📈 Plotando curva de aprendizado...")

# Obter histórico do treinamento
results = xgb_optimized.evals_result()

plt.figure(figsize=(12, 5))

# Loss durante o treinamento
plt.subplot(1, 2, 1)
plt.plot(results['train']['rmse'], label='Treino', linewidth=2)
plt.plot(results['val']['rmse'], label='Validação', linewidth=2)
plt.axvline(x=xgb_optimized.best_iteration, color='r', linestyle='--', label='Early Stop')
plt.xlabel('Iterações')
plt.ylabel('RMSE')
plt.title('Curva de Aprendizado - RMSE')
plt.legend()
plt.grid(True, alpha=0.3)

# Zoom na região do early stopping
plt.subplot(1, 2, 2)
start_idx = max(0, xgb_optimized.best_iteration - 200)
end_idx = min(len(results['train']['rmse']), xgb_optimized.best_iteration + 100)

plt.plot(range(start_idx, end_idx), results['train']['rmse'][start_idx:end_idx],
         label='Treino', linewidth=2)
plt.plot(range(start_idx, end_idx), results['val']['rmse'][start_idx:end_idx],
         label='Validação', linewidth=2)
plt.axvline(x=xgb_optimized.best_iteration, color='r', linestyle='--', label='Early Stop')
plt.xlabel('Iterações')
plt.ylabel('RMSE')
plt.title('Zoom - Região Early Stopping')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Análise de importância das features
print("📊 Analisando importância das features...")

# Importância por ganho (gain)
importance_gain = xgb_optimized.get_booster().get_score(importance_type='gain')
importance_df = pd.DataFrame({
    'feature': list(importance_gain.keys()),
    'importance': list(importance_gain.values())
}).sort_values('importance', ascending=False)

# Mapear os nomes das features (XGBoost usa f0, f1, etc.)
feature_names = X_train.columns.tolist()
importance_df['feature_name'] = importance_df['feature'].apply(
    lambda x: feature_names[int(x[1:])] if x.startswith('f') else x
)

# Top 20 features mais importantes
top_features = importance_df.head(20)

plt.figure(figsize=(12, 10))
sns.barplot(data=top_features, y='feature_name', x='importance')
plt.title('Top 20 Features Mais Importantes - XGBoost (Gain)')
plt.xlabel('Importância (Gain)')
plt.tight_layout()
plt.show()

print("🏆 Top 10 features mais importantes:")
for i, (_, row) in enumerate(top_features.head(10).iterrows(), 1):
    print(f"   {i:2d}. {row['feature_name']}: {row['importance']:.2f}")

# Salvar importância das features
importance_df.to_csv('feature_importance_xgb.csv', index=False)
print("\n💾 Importância das features salva em 'feature_importance_xgb.csv'")

In [ ]:
# Comparação de diferentes tipos de importância no XGBoost
print("🔍 Comparando diferentes tipos de importância...")

importance_types = ['gain', 'weight', 'cover']
importance_data = {}

for imp_type in importance_types:
    importance_dict = xgb_optimized.get_booster().get_score(importance_type=imp_type)
    importance_data[imp_type] = importance_dict

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, imp_type in enumerate(importance_types):
    # Converter para DataFrame
    imp_df = pd.DataFrame({
        'feature': list(importance_data[imp_type].keys()),
        'importance': list(importance_data[imp_type].values())
    }).sort_values('importance', ascending=False).head(10)

    # Mapear nomes das features
    imp_df['feature_name'] = imp_df['feature'].apply(
        lambda x: feature_names[int(x[1:])] if x.startswith('f') else x
    )

    # Plot
    sns.barplot(data=imp_df, y='feature_name', x='importance', ax=axes[i])
    axes[i].set_title(f'Importância por {imp_type.upper()}')
    axes[i].set_xlabel(f'Importância ({imp_type})')

plt.tight_layout()
plt.show()

In [ ]:
# Análise por estado - dados de teste
print("🗺️ Analisando performance por estado...")

# Criar DataFrame com resultados do teste
df_test = df[test_mask].copy()
df_test['Predicoes'] = y_pred_test_opt
df_test['Residuos'] = df_test['Predicoes'] - df_test['Quantidade de Casos']

# Métricas por estado
metrics_by_state = []
for state in df_test['COD_UF'].unique():
    state_data = df_test[df_test['COD_UF'] == state]
    if len(state_data) > 0:
        metrics = calculate_metrics(
            state_data['Quantidade de Casos'],
            state_data['Predicoes'],
            state
        )
        metrics['Estado'] = state
        metrics['N_Obs'] = len(state_data)
        metrics_by_state.append(metrics)

metrics_df = pd.DataFrame(metrics_by_state)

print("\n🏆 Top 5 estados com melhor R²:")
for _, row in metrics_df.nlargest(5, 'R²').iterrows():
    print(f"   {row['Estado']}: R² = {row['R²']:.3f}, RMSE = {row['RMSE']:.2f}")

print("\n🎯 5 estados com menor R²:")
for _, row in metrics_df.nsmallest(5, 'R²').iterrows():
    print(f"   {row['Estado']}: R² = {row['R²']:.3f}, RMSE = {row['RMSE']:.2f}")

# Salvar métricas por estado
metrics_df.to_csv('metricas_por_estado_xgb.csv', index=False)
print("\n💾 Métricas por estado salvas em 'metricas_por_estado_xgb.csv'")

In [ ]:
# Visualização das predições
def plot_predictions(y_true, y_pred, title="Predições vs Real"):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Scatter plot
    axes[0].scatter(y_true, y_pred, alpha=0.6)
    axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[0].set_xlabel('Valores Reais')
    axes[0].set_ylabel('Predições')
    axes[0].set_title(f'{title} - Scatter')

    # Residuos
    residuals = y_pred - y_true
    axes[1].scatter(y_pred, residuals, alpha=0.6)
    axes[1].axhline(y=0, color='r', linestyle='--')
    axes[1].set_xlabel('Predições')
    axes[1].set_ylabel('Resíduos')
    axes[1].set_title(f'{title} - Resíduos')

    plt.tight_layout()
    plt.show()

plot_predictions(y_test, y_pred_test_opt, "XGBoost Otimizado - Teste")

# Série temporal para alguns estados
estados_exemplo = ['SP', 'MG', 'RJ', 'BA', 'PR']

fig, axes = plt.subplots(len(estados_exemplo), 1, figsize=(15, 3*len(estados_exemplo)))

for i, estado in enumerate(estados_exemplo):
    state_data = df_test[df_test['COD_UF'] == estado].sort_values('Data')

    if len(state_data) > 0:
        axes[i].plot(state_data['Data'], state_data['Quantidade de Casos'],
                    label='Real', marker='o', linewidth=2)
        axes[i].plot(state_data['Data'], state_data['Predicoes'],
                    label='Predição', marker='s', linewidth=2, alpha=0.8)
        axes[i].set_title(f'Predições vs Real - {estado}')
        axes[i].set_ylabel('Casos de Dengue')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# SHAP Analysis (se disponível)
try:
    import shap
    print("🔍 Realizando análise SHAP...")

    # Criar explainer
    explainer = shap.TreeExplainer(xgb_optimized)

    # Calcular SHAP values para uma amostra dos dados de teste
    sample_size = min(1000, len(X_test))
    X_test_sample = X_test.sample(n=sample_size, random_state=42)
    shap_values = explainer.shap_values(X_test_sample)

    # Plot summary
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test_sample, feature_names=X_test.columns, show=False)
    plt.title('SHAP Summary Plot - XGBoost')
    plt.tight_layout()
    plt.show()

    print("✅ Análise SHAP concluída!")

except ImportError:
    print("⚠️ SHAP não instalado. Pulando análise SHAP.")
except Exception as e:
    print(f"⚠️ Erro na análise SHAP: {e}")

In [ ]:
# Salvar modelo e resultados
print("💾 Salvando modelo treinado...")

# Salvar o modelo
joblib.dump(xgb_optimized, 'modelo_xgboost.pkl')
print("✅ Modelo salvo: 'modelo_xgboost.pkl'")

# Salvar hiperparâmetros
with open('hiperparametros_xgb.txt', 'w', encoding='utf-8') as f:
    f.write("HIPERPARÂMETROS OTIMIZADOS - XGBOOST\n")
    f.write("="*50 + "\n\n")
    for param, value in random_search.best_params_.items():
        f.write(f"{param}: {value}\n")
    f.write(f"\nbest_iteration: {xgb_optimized.best_iteration}\n")

print("✅ Hiperparâmetros salvos: 'hiperparametros_xgb.txt'")

# Compilar todas as métricas
all_metrics = [
    metrics_train_baseline, metrics_val_baseline,
    metrics_train_opt, metrics_val_opt, metrics_test_opt
]

metrics_comparison = pd.DataFrame(all_metrics)
metrics_comparison.to_csv('metricas_comparacao_xgb.csv', index=False)
print("✅ Comparação de métricas salva: 'metricas_comparacao_xgb.csv'")

# Salvar predições do teste
test_predictions = pd.DataFrame({
    'Real': y_test,
    'Predicao': y_pred_test_opt,
    'Residuo': y_pred_test_opt - y_test
})
test_predictions.to_csv('predicoes_teste_xgb.csv', index=False)
print("✅ Predições do teste salvas: 'predicoes_teste_xgb.csv'")

In [ ]:
# Comparação com Random Forest (se disponível)
try:
    rf_metrics = pd.read_csv('metricas_comparacao_rf.csv')
    xgb_metrics = pd.read_csv('metricas_comparacao_xgb.csv')

    print("📊 COMPARAÇÃO: XGBoost vs Random Forest")
    print("="*50)

    # Pegar métricas de teste
    rf_test = rf_metrics[rf_metrics['Modelo'].str.contains('Teste')].iloc[0] if len(rf_metrics[rf_metrics['Modelo'].str.contains('Teste')]) > 0 else None
    xgb_test = xgb_metrics[xgb_metrics['Modelo'].str.contains('Teste')].iloc[0]

    if rf_test is not None:
        print(f"\n🎯 Métricas no conjunto de TESTE:")
        print(f"                    Random Forest    XGBoost      Diferença")
        print(f"   RMSE:              {rf_test['RMSE']:8.2f}   {xgb_test['RMSE']:8.2f}   {((xgb_test['RMSE'] - rf_test['RMSE'])/rf_test['RMSE']*100):+8.2f}%")
        print(f"   MAE:               {rf_test['MAE']:8.2f}   {xgb_test['MAE']:8.2f}   {((xgb_test['MAE'] - rf_test['MAE'])/rf_test['MAE']*100):+8.2f}%")
        print(f"   R²:                {rf_test['R²']:8.4f}   {xgb_test['R²']:8.4f}   {((xgb_test['R²'] - rf_test['R²'])/rf_test['R²']*100):+8.2f}%")
        print(f"   MAPE:              {rf_test['MAPE']:8.2f}   {xgb_test['MAPE']:8.2f}   {((xgb_test['MAPE'] - rf_test['MAPE'])/rf_test['MAPE']*100):+8.2f}%")

        # Determinar melhor modelo
        if xgb_test['R²'] > rf_test['R²']:
            print("\n🏆 XGBoost apresenta melhor performance!")
        else:
            print("\n🏆 Random Forest apresenta melhor performance!")

except FileNotFoundError:
    print("⚠️ Arquivo de métricas do Random Forest não encontrado.")
except Exception as e:
    print(f"⚠️ Erro na comparação: {e}")

In [ ]:
# Resumo final
print("🎯 RESUMO - XGBOOST")
print("="*50)
print(f"\n📊 Modelo Final:")
print(f"   Algoritmo: XGBoost Regressor")
for param, value in random_search.best_params_.items():
    print(f"   {param}: {value}")
print(f"   best_iteration: {xgb_optimized.best_iteration}")

print(f"\n📈 Performance (Teste):")
print(f"   RMSE: {metrics_test_opt['RMSE']:.2f}")
print(f"   MAE: {metrics_test_opt['MAE']:.2f}")
print(f"   R²: {metrics_test_opt['R²']:.4f}")
print(f"   MAPE: {metrics_test_opt['MAPE']:.2f}%")

print(f"\n🏆 Top 3 Features Mais Importantes:")
for i, (_, row) in enumerate(top_features.head(3).iterrows(), 1):
    print(f"   {i}. {row['feature_name']}: {row['importance']:.2f}")

print(f"\n🗺️ Performance por Região:")
print(f"   Melhor estado (R²): {metrics_df.loc[metrics_df['R²'].idxmax(), 'Estado']} ({metrics_df['R²'].max():.3f})")
print(f"   Pior estado (R²): {metrics_df.loc[metrics_df['R²'].idxmin(), 'Estado']} ({metrics_df['R²'].min():.3f})")
print(f"   R² médio: {metrics_df['R²'].mean():.3f} ± {metrics_df['R²'].std():.3f}")

print(f"\n🛑 Early Stopping:")
print(f"   Melhor iteração: {xgb_optimized.best_iteration}")
print(f"   Total de estimators: {xgb_optimized.n_estimators}")

print("\n✅ XGBoost - Treinamento Concluído!")
print("Próximo: Treinamento LightGBM")